In [115]:
import numpy as np
import pandas as pd
import os

In [116]:
class Node:
    def __init__(self, feature_name=None, threshold=None, value=None):
        self.feature_name  = feature_name
        self.threshold     = threshold
        self.left          = None
        self.right         = None
        self.value         = value

In [ ]:
class DecisionTreeClassifier:
    def __init__(self, X, y, max_depth=None):
        self.feature_names = X.columns
        self.max_depth = max_depth
        self.X = X
        self.y = y
        self.root = None

    def print_tree(self, node, depth=0):
        indent = "  " * depth  # Create indentation based on depth
        # If it's a leaf node, print its value
        if node.value is not None:
            print(f"{indent}Leaf: {node.value}")     
        else:
            # Otherwise, print the decision rule and recursively print children
            print(f"{indent}Node: if {node.feature_name} <= {node.threshold}?")
            print(f"{indent}Left:")
            self.print_tree(node.left, depth + 1)
            print(f"{indent}Right:")
            self.print_tree(node.right, depth + 1)
    
    def fit(self):
        self.root = self._build_tree(self.X, self.y)

    def _build_tree(self, X, y, depth=0):

        if y.nunique() == 1:
            return Node(value=y.iloc[0])

        if self.max_depth and (depth >= self.max_depth):
            return Node(value=y.value_counts().idxmax())
        
        best_split_feature, best_split_value = self._best_split(X, y)
        X_left, y_left, X_right, y_right = self._split_values(X, y, best_split_feature, best_split_value)
        node = Node(best_split_feature, best_split_value)
      
        node.left  = self._build_tree(X_left, y_left, depth+1)
        node.right = self._build_tree(X_right, y_right, depth+1)
        
        return node
    
    def _gini(self, distribution):
        """ Calculate a given distribution's gini impurity """
        counts = distribution.value_counts().values
        probabilities = counts / len(distribution)
        gini = 1 - sum(probabilities ** 2)
        return gini

    def _split_values(self, X, y, feature, threshold):
        """ Split the dataframe (X and y) into two parts based on whether the value of `feature` is smaller (or equal) or greater than `threshold` """
        left_mask = X[feature] <= threshold
        X_left = X[left_mask]
        y_left = y[left_mask]

        X_right = X[~left_mask]
        y_right  = y[~left_mask]

        return X_left, y_left, X_right, y_right
    
    def _best_split(self, X, y):
        """ Determine the split with the lowest gini impurity, this will be the best split """
        
        best_split_feature = None
        best_split_value = None
        lowest_gini_impurity = float('inf')

        for feature in self.feature_names:

            for possible_threshold in X[feature]:
                X_left, y_left, X_right, y_right = self._split_values(X, y, feature, possible_threshold)
                gini_left = self._gini(y_left)
                gini_right = self._gini(y_right)

                total_y_count = len(y_left) + len(y_right)

                weighted_gini = len(y_left) / total_y_count * gini_left + len(y_right) / total_y_count * gini_right

                if weighted_gini < lowest_gini_impurity:
                    best_split_feature = feature
                    best_split_value = possible_threshold
                    lowest_gini_impurity = weighted_gini

        return best_split_feature, best_split_value

    def predict(self, sample):
        return self._predict_recur(sample, self.root)

    def _predict_recur(self, sample, node):
        curr_feature_name = node.feature_name
        curr_threshold = node.threshold

        if node.value is not None:
            return node.value

        if sample[curr_feature_name] <= curr_threshold:
            return self._predict_recur(sample, node.left)
        else:
            return self._predict_recur(sample, node.right)
        
        

In [ ]:
class randomForest:
    # TO BE IMPLEMENTED
    pass

In [118]:
def split_train_test(data):
    data_train = data.sample(frac=0.8, random_state=42)
    data_test = data.drop(data_train.index)
    return data_train, data_test

In [119]:
data = pd.read_csv("IRIS.csv")
data_train, data_test = split_train_test(data)

X_train = data_train.iloc[:, :-1]
y_train = data_train.iloc[:, -1]

X_test = data_test.iloc[:, :-1]
y_test = data_test.iloc[:, -1]

tree = DecisionTreeClassifier(X_train, y_train)
tree.fit()
# tree.print_tree(tree.root)
print(tree.predict(X_test.iloc[2]))
print(f"Correct result: {y_test.iloc[2]}")

Iris-setosa
Correct result: Iris-setosa


In [120]:
data[data["petal_length"] > 1.9]

,sepal_length,sepal_width,petal_length,petal_width,species
50,7.0,3.2,4.7,1.4,Iris-versicolor
51,6.4,3.2,4.5,1.5,Iris-versicolor
52,6.9,3.1,4.9,1.5,Iris-versicolor
53,5.5,2.3,4.0,1.3,Iris-versicolor
54,6.5,2.8,4.6,1.5,Iris-versicolor
...,...,...,...,...,...
145,6.7,3.0,5.2,2.3,Iris-virginica
146,6.3,2.5,5.0,1.9,Iris-virginica
147,6.5,3.0,5.2,2.0,Iris-virginica
148,6.2,3.4,5.4,2.3,Iris-virginica
